In [31]:
import requests
from bs4 import BeautifulSoup
import csv
import time

BASE_URL = "https://books.toscrape.com/catalogue/"
START_URL = "https://books.toscrape.com/catalogue/page-1.html"

def get_rating(word):
    ratings = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}
    return ratings.get(word, 0)

def scrape_books(max_pages=5):
    books = []
    url = START_URL

    for page in range(1, max_pages + 1):
        print(f"Scraping page {page}...")
        response = requests.get(url)
        
        if response.status_code != 200:
            print(f"Failed to fetch page {page}")
            break

        soup = BeautifulSoup(response.text, "html.parser")
        articles = soup.find_all("article", class_="product_pod")

        for article in articles:
            title = article.h3.a["title"]
            price = article.find("p", class_="price_color").text.strip()
            rating_word = article.p["class"][1]
            rating = get_rating(rating_word)
            availability = article.find("p", class_="instock availability").text.strip()

            books.append({
                "Title": title,
                "Price (£)": price.replace("£", "").replace("Â", ""),
                "Rating": rating,
                "Availability": availability
            })

        # Find next page
        next_btn = soup.find("li", class_="next")
        if next_btn:
            next_page = next_btn.a["href"]
            url = BASE_URL + next_page
        else:
            print("No more pages.")
            break

        time.sleep(1)  # polite delay

    return books

def save_to_csv(books, filename="books_dataset.csv"):
    if not books:
        print("No data to save.")
        return

    with open(filename, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=books[0].keys())
        writer.writeheader()
        writer.writerows(books)

    print(f"\n✅ Dataset saved: {filename}")
    print(f"📚 Total books scraped: {len(books)}")

def analyze_data(books):
    print("\n📊 Basic Analysis:")
    prices = [float(b["Price (£)"]) for b in books if b["Price (£)"]]
    ratings = [b["Rating"] for b in books]

    print(f"  Average Price : £{sum(prices)/len(prices):.2f}")
    print(f"  Cheapest Book : £{min(prices):.2f}")
    print(f"  Most Expensive: £{max(prices):.2f}")
    print(f"  Average Rating: {sum(ratings)/len(ratings):.1f} / 5")

    print("\n⭐ Rating Distribution:")
    for r in range(1, 6):
        count = ratings.count(r)
        bar = "█" * count
        print(f"  {r} star: {bar} ({count})")

# --- Main ---
books = scrape_books(max_pages=5)
save_to_csv(books)
analyze_data(books)

# Preview first 3 entries
print("\n📋 Sample Data:")
for b in books[:3]:
    print(b)

Scraping page 1...
Scraping page 2...
Scraping page 3...
Scraping page 4...
Scraping page 5...

✅ Dataset saved: books_dataset.csv
📚 Total books scraped: 100

📊 Basic Analysis:
  Average Price : £34.56
  Cheapest Book : £10.16
  Most Expensive: £58.11
  Average Rating: 2.9 / 5

⭐ Rating Distribution:
  1 star: ██████████████████████ (22)
  2 star: ███████████████████ (19)
  3 star: ██████████████████████ (22)
  4 star: ██████████████████ (18)
  5 star: ███████████████████ (19)

📋 Sample Data:
{'Title': 'A Light in the Attic', 'Price (£)': '51.77', 'Rating': 3, 'Availability': 'In stock'}
{'Title': 'Tipping the Velvet', 'Price (£)': '53.74', 'Rating': 1, 'Availability': 'In stock'}
{'Title': 'Soumission', 'Price (£)': '50.10', 'Rating': 1, 'Availability': 'In stock'}
